In [0]:
%sql

SELECT *
FROM spotifyde_catalog.silver.track

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,medium
2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,medium
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,medium
4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,medium
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,medium
6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,high
7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,medium
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,medium
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,high
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,medium


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS spotifyde_catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS spotifyde_catalog.gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotifyde_catalog.gold.dim_track_scd2
USING DELTA
LOCATION "abfss://gold@adlspotifyde.dfs.core.windows.net/scd2/dim_track_scd2"
AS
SELECT 
    row_number() over (order by track_id) AS DimTrackKey_scd2,
    track_id,
    track_name,
    artist_id,
    album_name,
    duration_sec,
    release_date,
    updated_at,

 current_date() AS effective_start_date,
 CAST (NULL AS DATE) as effective_end_date,
 TRUE AS is_current
 
 FROM spotifyde_catalog.silver.track;



num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM spotifyde_catalog.gold.dim_track_scd2

DimTrackKey_scd2,track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,effective_start_date,effective_end_date,is_current
1,1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,2026-09-15,null,true
2,2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,2026-09-15,null,true
3,3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,2026-09-15,null,true
4,4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,2026-09-15,null,true
5,5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,2026-09-15,null,true
6,6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,2026-09-15,null,true
7,7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,2026-09-15,null,true
8,8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,2026-09-15,null,true
9,9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,2026-09-15,null,true
10,10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,2026-09-15,null,true


In [0]:
%sql
-- Close the Old version
-- <=> handles NULL correctly.

MERGE INTO spotifyde_catalog.gold.dim_track_scd2 AS target
USING spotifyde_catalog.silver.track AS source

ON target.track_id = source.track_id
AND target.is_current = TRUE

WHEN MATCHED
AND (
       NOT (target.track_name <> source.track_name)
    OR NOT (target.artist_id <> source.artist_id)
    OR NOT (target.album_name <> source.album_name)
    OR NOT (target.duration_sec <> source.duration_sec)
    OR NOT (target.release_date <> source.release_date)
    OR NOT (target.updated_at <> source.updated_at)
)
AND target.effective_end_date IS NULL

THEN UPDATE SET
    target.effective_end_date = current_date(),
    target.is_current = FALSE;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
500,500,0,0


In [0]:
%sql
-- Handle New Version

INSERT INTO spotifyde_catalog.gold.dim_track_scd2
(
    DimTrackKey_scd2,
    track_id,
    track_name,
    artist_id,
    album_name,
    duration_sec,
    release_date,
    updated_at,
    effective_start_date,
    effective_end_date,
    is_current
)

SELECT
    row_number() OVER (ORDER BY source.track_id)
        + COALESCE(
            (
                SELECT MAX(DimTrackKey_scd2)
                FROM spotifyde_catalog.gold.dim_track_scd2
            ),
            0
          ) AS DimTrackKey_scd2,

    source.track_id,
    source.track_name,
    source.artist_id,
    source.album_name,
    source.duration_sec,
    source.release_date,
    source.updated_at,

    current_date() AS effective_start_date,
    CAST(NULL AS DATE) AS effective_end_date,
    TRUE AS is_current

FROM spotifyde_catalog.silver.track AS source

LEFT JOIN spotifyde_catalog.gold.dim_track_scd2 AS target
    ON source.track_id = target.track_id
    AND target.is_current = TRUE

WHERE target.track_id IS NULL;


num_affected_rows,num_inserted_rows
500,500


## Test SCD2 with below sample

In [0]:
%sql
SELECT *
FROM spotifyde_catalog.gold.dim_track_scd2
WHERE track_id = 101
ORDER BY effective_start_date;

DimTrackKey_scd2,track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,effective_start_date,effective_end_date,is_current
101,101,Mandatory even keeled moderator,484,Executive Album,107,2021-03-16,2025-09-15T19:49:55.000Z,2026-09-15,2026-09-15,false
601,101,Mandatory even keeled moderator,484,Executive Album,107,2021-03-16,2025-09-15T19:49:55.000Z,2026-09-15,null,true


In [0]:
%sql
SELECT *
FROM spotifyde_catalog.gold.dim_track_scd2
WHERE track_id = 1
ORDER BY effective_start_date;

DimTrackKey_scd2,track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,effective_start_date,effective_end_date,is_current
1,1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,2026-09-15,2026-09-15,false
501,1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,2026-09-15,null,true
